[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/padillacm/IA-practice/blob/master/tokenization-embeddings-course/05_patologias_robustez/05_proyecto_patologias_robustez.ipynb)

# Proyecto 05 · Detector de tokens sub-entrenados y *fuzzing* de tokenizer + modelo

| | |
|---|---|
| **Nivel** | 🟠 Avanzado |
| **Duración** | ≈ 2 h |
| **GPU** | Ninguna (CPU, `FAST_DEV_RUN = True`: banco MiniGPT ≈ 1 min). Con GPT-2 real y red: CPU basta |
| **Prerrequisitos** | Lección del módulo 05 y proyecto 04 |

## Contexto de negocio
Antes de congelar el vocabulario del embedder, el CTO de Políglota Labs te pide **un informe de robustez** con (A) un **detector de tokens sub-entrenados** sobre un modelo pequeño, (B) un **sanitizador de Unicode invisible** para indexar documentos de clientes y (C) un **fuzzer** que pruebe tokenizer + modelo y liste hallazgos con severidad y mitigación.
Es trabajo *defensivo*: no se usan payloads dañinos (solo texto benigno).

> **Conexión.** Viene del proyecto 04 (receta de vocabulario) y alimenta el embedder de los módulos 09–11 (el vocabulario que congeles hereda sus tokens basura). Va hacia el módulo 06 (modelos sin tokenizer: qué patologías desaparecen).

## Dataset y banco de pruebas
Corpus Políglota (`carga_corpus`: HF → espejo Europarl + stdlib → sintético). El **banco** (`cargar_banco`) entrena un MiniGPT con embeddings atados cuyo tokenizer vio un «hilo de nombres de usuario» que el modelo nunca ve: conocemos la **verdad** (frecuencia 0) y los 40 *intrusos* para medir tu detector. Con `FAST_DEV_RUN=False` y red usa GPT-2 real (sin verdad de referencia: solo ranking).

## Entregables y rúbrica
1. `ind_coseno` y `ind_norma` (indicadores de pesos), `candidatos(banco, k)` y `confirma(banco, ids)` (verificación por comportamiento).
2. `inspecciona` y `sanitiza` (Unicode invisible) que superen los tests unitarios.
3. `fuzz_props(banco, ...)` con 5 propiedades y un `informe` con ≥ 3 hallazgos (severidad + mitigación).

Los umbrales de la rúbrica se calibraron con **lo que mide la solución de referencia** en el banco MiniGPT (precisión@100 ≈ 1.00 → umbral 0.90; recall de intrusos en el top-100 ≈ 0.42, porque hay ≈ 300 tokens no usados y 40 intrusos → umbral 0.30; P5 en la clase `glitch` ≈ 83 % → umbral 60 %; 6/6 tests de sanitización) menos un margen; con GPT-2 real solo se evalúan los criterios sin verdad de referencia.

In [ ]:
!pip install -q -U tokenizers sentencepiece tiktoken datasets matplotlib seaborn scikit-learn

In [ ]:
import os, sys, re, math, json, random, time, unicodedata, collections, itertools, warnings
os.environ.setdefault("OMP_NUM_THREADS", "1")          # evita sobre-suscripción de hilos
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
warnings.filterwarnings("ignore")
import numpy as np
import matplotlib.pyplot as plt

SEED = 42
FAST_DEV_RUN = True      # True: tamaños diminutos (CPU, minutos). False: escala completa (Colab Pro)
random.seed(SEED); np.random.seed(SEED)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25, "axes.spines.top": False, "axes.spines.right": False})
COL = {"azul": "#1F6FEB", "naranja": "#F28E2B", "verde": "#2CA02C", "gris": "#6B7280", "rojo": "#D62728"}
try:
    import torch
    torch.set_num_threads(1); torch.manual_seed(SEED)
    device = "cuda" if torch.cuda.is_available() else "cpu"
except ImportError:
    device = "cpu"
print("device:", device, "| FAST_DEV_RUN:", FAST_DEV_RUN)

In [ ]:
# ---- Utilidades de corpus Políglota (copia local; contrato: carga_corpus(fast_dev) -> dict) ----
import urllib.request, urllib.error, zipfile, io, inspect, ast

def hay_red(url: str, timeout: float = 6.0) -> bool:
    """Sonda rápida para no esperar minutos si el host está bloqueado (proxy corporativo, etc.)."""
    try:
        urllib.request.urlopen(urllib.request.Request(url, method="HEAD"), timeout=timeout)
        return True
    except urllib.error.HTTPError:
        return True          # el servidor respondió (aunque sea 4xx): hay ruta de red
    except Exception:
        return False         # túnel bloqueado, DNS, timeout...

def _hf_wikipedia(lang: str, n: int, max_chars: int):
    from datasets import load_dataset
    ds = load_dataset("wikimedia/wikipedia", f"20231101.{lang}", split="train", streaming=True)
    out = []
    for ex in ds:
        if len(ex["text"]) > 400:
            out.append({"id": f"wiki_{lang}_{len(out)}", "lang": lang, "text": ex["text"][:max_chars]})
        if len(out) >= n:
            break
    return out

def _hf_codesearchnet(n: int):
    from datasets import load_dataset
    for repo, kw in [("code-search-net/code_search_net", {"name": "python"}), ("claudios/code_search_net", {"name": "python"}),
                     ("Nan-Do/code-search-net-python", {})]:
        try:
            ds = load_dataset(repo, split="train", streaming=True, **kw)
            out = []
            for ex in ds:
                code = ex.get("func_code_string") or ex.get("code") or ex.get("original_string")
                if code and 80 < len(code) < 3000:
                    out.append({"id": f"csn_{len(out)}", "lang": "code", "text": code})
                if len(out) >= n:
                    return out
        except Exception as e:
            print("   (CodeSearchNet vía", repo, "falló:", type(e).__name__, ")")
    raise RuntimeError("CodeSearchNet no disponible")

def _detok(s: str) -> str:
    s = re.sub(r"\s+([,.;:!?%)\]])", r"\1", s)
    s = re.sub(r"([(\[¿¡])\s+", r"\1", s)
    return re.sub(r"\s+", " ", s).strip()

In [ ]:
def _espejo_nltk(n: int):
    """Espejo público: muestra de Europarl (ES/EN) del repositorio nltk_data en GitHub (CC/uso de investigación)."""
    url = "https://raw.githubusercontent.com/nltk/nltk_data/gh-pages/packages/corpora/europarl_raw.zip"
    z = zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(url, timeout=60).read()))
    docs = []
    for lang, folder, ext in [("es", "spanish", ".es"), ("en", "english", ".en")]:
        lines = []
        for name in sorted(z.namelist()):
            if f"/{folder}/" in name and name.endswith(ext):
                lines += [_detok(l) for l in z.read(name).decode("utf-8", "ignore").splitlines() if len(l.split()) > 6]
        step = 8
        chunks = [" ".join(lines[i:i + step]) for i in range(0, len(lines), step)]
        random.Random(SEED).shuffle(chunks)
        docs += [{"id": f"ep_{lang}_{i}", "lang": lang, "text": c[:2000]} for i, c in enumerate(chunks[:n])]
    return docs

def _codigo_stdlib(n: int):
    """Código Python real de la biblioteca estándar (siempre disponible, sin red)."""
    import json, re as _re, argparse, textwrap, collections as _c, itertools as _i, functools, pathlib, csv, heapq, bisect
    import statistics, difflib, string, shutil, tempfile, fractions, random as _r, datetime, calendar, glob, fnmatch, logging
    out = []
    for mod in [json.decoder, json.encoder, _re, argparse, textwrap, functools, pathlib, csv, heapq, bisect, statistics, difflib,
                shutil, tempfile, fractions, datetime, calendar, glob, fnmatch, logging, _c, _r]:
        try:
            tree = ast.parse(inspect.getsource(mod)); src = inspect.getsource(mod).splitlines()
        except (OSError, TypeError, SyntaxError):
            continue
        for node in ast.walk(tree):
            if isinstance(node, (ast.FunctionDef,)) and 4 <= (node.end_lineno - node.lineno) <= 40:
                code = "\n".join(src[node.lineno - 1: node.end_lineno])
                if 80 < len(code) < 3000:
                    out.append({"id": f"std_{len(out)}", "lang": "code", "text": code})
    random.Random(SEED).shuffle(out)
    return out[:n]

In [ ]:
_ES_S = ["el sistema", "la base de datos", "el servidor", "la aplicación", "el modelo", "la biblioteca", "el usuario", "la función", "el módulo", "la consulta"]
_ES_V = ["procesa", "almacena", "devuelve", "actualiza", "valida", "descarga", "ejecuta", "registra", "indexa", "compara"]
_ES_O = ["los documentos relevantes", "una lista de resultados", "los parámetros de configuración", "las peticiones entrantes", "la información almacenada",
         "varias tablas de forma eficiente", "los errores más frecuentes", "las versiones anteriores", "los textos en diferentes idiomas", "un conjunto de datos nuevo"]
_ES_A = ["rápidamente", "automáticamente", "correctamente", "de manera incremental", "sin interrupciones", "cada vez que cambia la configuración", "antes de la instalación", "durante la inicialización"]
_EN_S = ["the system", "the database", "the server", "the application", "the model", "the library", "the user", "the function", "the module", "the query"]
_EN_V = ["processes", "stores", "returns", "updates", "validates", "downloads", "executes", "logs", "indexes", "compares"]
_EN_O = ["the relevant documents", "a list of results", "the configuration parameters", "the incoming requests", "the stored information",
         "several tables efficiently", "the most frequent errors", "the previous versions", "texts in different languages", "a new dataset"]
_EN_A = ["quickly", "automatically", "correctly", "incrementally", "without interruptions", "whenever the configuration changes", "before the installation", "during initialization"]

def _sintetico(n: int):
    """Último recurso: texto ES/EN generado por plantillas + funciones Python plantilla (mismo esquema)."""
    rng = random.Random(SEED); docs = []
    for lang, S, V, O, A in [("es", _ES_S, _ES_V, _ES_O, _ES_A), ("en", _EN_S, _EN_V, _EN_O, _EN_A)]:
        for i in range(n):
            sents = []
            for _ in range(rng.randint(4, 8)):
                s = f"{rng.choice(S)} {rng.choice(V)} {rng.choice(O)} {rng.choice(A)}."
                sents.append(s[0].upper() + s[1:])
            docs.append({"id": f"syn_{lang}_{i}", "lang": lang, "text": " ".join(sents)})
    names = ["load", "parse", "merge", "filter", "count", "split", "encode", "decode", "score", "rank"]
    things = ["items", "tokens", "rows", "words", "records", "paths", "values", "pairs"]
    for i in range(n):
        f, t = rng.choice(names), rng.choice(things)
        docs.append({"id": f"syn_code_{i}", "lang": "code", "text":
            f"def {f}_{t}({t}, limit={rng.randint(2, 99)}):\n    \"\"\"{f.capitalize()} the {t} and return at most limit results.\"\"\"\n"
            f"    result = []\n    for x in {t}:\n        if len(result) >= limit:\n            break\n        result.append(x)\n    return result\n"})
    return docs

In [ ]:
def _mini_golden(docs, n_q: int = 24):
    """Mini golden set: la consulta es la primera frase (≤ 12 palabras) del documento; qrels {qid: {doc_id: 1}}."""
    rng = random.Random(SEED); queries, qrels = [], {}
    for lang in ["es", "en", "code"]:
        pool = [d for d in docs if d["lang"] == lang]; rng.shuffle(pool)
        for d in pool[: n_q // 3]:
            if lang == "code":
                try:
                    q = (ast.get_docstring(ast.parse(d["text"]).body[0]) or "").split("\n")[0]
                except SyntaxError:
                    q = ""
                q = q or re.sub(r"[_(].*", "", d["text"].split("def ")[-1]).replace("_", " ")
            else:
                q = " ".join(re.split(r"(?<=[.!?])\s", d["text"])[0].split()[:12])
            qid = f"q{len(queries)}"; queries.append({"qid": qid, "lang": lang, "text": q}); qrels[qid] = {d["id"]: 1}
    return queries, qrels

In [ ]:
def carga_corpus(fast_dev: bool = True) -> dict:
    """Corpus Políglota. Cadena: oficial (HF Wikipedia + CodeSearchNet) -> espejo (Europarl vía GitHub + stdlib) -> sintético.
    Devuelve {"docs", "queries", "qrels", "source": "real"|"synthetic", "detail": {...}}."""
    n = 150 if fast_dev else 3000
    max_chars = 2000 if fast_dev else 6000
    docs, detail = [], {}
    hf_ok = hay_red("https://huggingface.co")
    for lang in ["es", "en"]:
        got = None
        if hf_ok:
            try: got, detail[lang] = _hf_wikipedia(lang, n, max_chars), "hf:wikimedia/wikipedia"
            except Exception as e: print(f"   (Wikipedia {lang} falló: {type(e).__name__})")
        if got is None and hay_red("https://raw.githubusercontent.com"):
            try:
                mirror = _espejo_nltk(n); got = [d for d in mirror if d["lang"] == lang]; detail[lang] = "espejo:nltk-europarl"
            except Exception as e: print(f"   (espejo {lang} falló: {type(e).__name__})")
        if got is None: got, detail[lang] = [d for d in _sintetico(n) if d["lang"] == lang], "sintetico"
        docs += got
    code = None
    if hf_ok:
        try: code, detail["code"] = _hf_codesearchnet(n), "hf:code_search_net"
        except Exception as e: print(f"   (CodeSearchNet falló: {type(e).__name__})")
    if code is None:
        code = _codigo_stdlib(n); detail["code"] = "espejo:stdlib-python"
        if len(code) < 10: code, detail["code"] = [d for d in _sintetico(n) if d["lang"] == "code"], "sintetico"
    docs += code
    queries, qrels = _mini_golden(docs)
    real = all(not v.startswith("sintetico") for v in detail.values())
    if not real: print("AVISO: parte del corpus es SINTÉTICA (sin acceso a los datasets). Las cifras absolutas cambiarán con datos reales.")
    return {"docs": docs, "queries": queries, "qrels": qrels, "source": "real" if real else "synthetic", "detail": detail}

In [ ]:
corpus = carga_corpus(fast_dev=FAST_DEV_RUN)
docs = corpus["docs"]
print("fuente:", corpus["source"], corpus["detail"])
by_lang = {l: [d["text"] for d in docs if d["lang"] == l] for l in ["es", "en", "code"]}
for l, ts in by_lang.items():
    print(f"{l:>4}: {len(ts):4d} docs | {sum(len(t) for t in ts)/1e3:8.1f} k caracteres | {sum(len(t.encode()) for t in ts)/1e3:8.1f} k bytes")
print("\nEjemplo ES:", by_lang["es"][0][:160].replace("\n", " "))
print("Ejemplo código:\n", by_lang["code"][0][:200])

In [ ]:
import tiktoken
TIK = {}
for nombre in ["r50k_base", "cl100k_base", "o200k_base"]:
    try:
        TIK[nombre] = tiktoken.get_encoding(nombre)     # 1ª vez: descarga de openaipublic.blob.core.windows.net
    except Exception as e:
        print(f"(no pude cargar {nombre}: {type(e).__name__}; tras un proxy usa TIKTOKEN_CACHE_DIR). Se omitirá en los análisis.")
print({n: e.n_vocab for n, e in TIK.items()})

In [ ]:
import pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score
import unicodedata
import torch, torch.nn as nn, torch.nn.functional as F
from tokenizers import Tokenizer, models, pre_tokenizers, trainers, decoders, Regex
from tokenizers.pre_tokenizers import ByteLevel
from collections import Counter

PAT_GPT2 = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""

def bytes_a_unicode() -> dict:
    """Tabla de GPT-2: byte 0-255 -> carácter imprimible (así se escriben los tokens byte-level en los vocabularios HF)."""
    bs = list(range(ord("!"), ord("~") + 1)) + list(range(ord("¡"), ord("¬") + 1)) + list(range(ord("®"), ord("ÿ") + 1))
    cs, n = bs[:], 0
    for b in range(256):
        if b not in bs: bs.append(b); cs.append(256 + n); n += 1
    return dict(zip(bs, map(chr, cs)))
B2U = bytes_a_unicode(); U2B = {v: k for k, v in B2U.items()}
def token_a_bytes(t: str) -> bytes: return bytes(U2B[ch] for ch in t if ch in U2B)
def token_a_texto(t: str) -> str: return token_a_bytes(t).decode("utf-8", "replace")
BYTES_INVALIDOS = list(range(0xF5, 0x100))          # estos bytes NUNCA aparecen en UTF-8 válido: sus tokens son "no usados" de referencia

In [ ]:
class _Bloque(nn.Module):
    def __init__(s, d, h):
        super().__init__(); s.ln1 = nn.LayerNorm(d); s.att = nn.MultiheadAttention(d, h, batch_first=True); s.ln2 = nn.LayerNorm(d)
        s.mlp = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Linear(4 * d, d))
    def forward(s, x, m):
        a = s.ln1(x); x = x + s.att(a, a, a, attn_mask=m, need_weights=False)[0]; return x + s.mlp(s.ln2(x))

class MiniGPT(nn.Module):
    """GPT diminuto con embeddings ATADOS (como GPT-2): la misma matriz sirve de entrada y de salida."""
    def __init__(s, V, d=64, L=2, h=4, ctx=64):
        super().__init__(); s.emb = nn.Embedding(V, d); s.pos = nn.Embedding(ctx, d); s.ctx = ctx
        s.blocks = nn.ModuleList([_Bloque(d, h) for _ in range(L)]); s.ln = nn.LayerNorm(d); nn.init.normal_(s.emb.weight, std=0.02)
    def forward(s, x):
        T = x.shape[1]; m = torch.triu(torch.ones(T, T, dtype=torch.bool), 1); h = s.emb(x) + s.pos(torch.arange(T))
        for b in s.blocks: h = b(h, m)
        return s.ln(h) @ s.emb.weight.T

SILABAS = ["Mag", "ika", "Gold", "Troll", "Stream", "Bot", "Nitro", "Fan", "Redd", "Pixel", "Quan", "Zed", "Moon", "Karp", "Sol"]
def nombres_intrusos(n: int = 40, semilla: int = SEED) -> list:
    r = random.Random(semilla); return sorted({" " + "".join(r.choice(SILABAS) for _ in range(3)) for _ in range(n * 2)})[:n]

def banco_mini(docs, vocab_size: int = 2000, pasos: int = 500, semilla: int = SEED) -> dict:
    """Banco de pruebas reproducible del fenómeno 'glitch token':
    - el TOKENIZER se entrena con el corpus + un 'hilo de nombres de usuario' (los INTRUSOS, muy repetidos)
    - el MODELO se entrena SOLO con el corpus (nunca ve a los intrusos)  =>  los intrusos son tokens sub-entrenados.
    Devuelve un 'bundle' con la misma interfaz que usaremos para GPT-2 real."""
    torch.manual_seed(semilla); intr = nombres_intrusos()
    textos = [d["text"] for d in docs]
    hilo = [f"u/{n.strip()} posted: {n} {n} {n}\n" * 3 for n in intr for _ in range(120)]
    tk = Tokenizer(models.BPE())
    tk.pre_tokenizer = pre_tokenizers.Sequence([pre_tokenizers.Split(Regex(PAT_GPT2), behavior="isolated"), ByteLevel(add_prefix_space=False, use_regex=False)])
    tk.decoder = decoders.ByteLevel()
    tk.train_from_iterator(textos + hilo, trainers.BpeTrainer(vocab_size=vocab_size, initial_alphabet=ByteLevel.alphabet(), special_tokens=["<|endoftext|>"]))
    V = tk.get_vocab_size(); ids = []
    for t in textos: ids += tk.encode(t).ids + [0]
    ids = torch.tensor(ids); cnt = Counter(ids.tolist())
    model = MiniGPT(V); opt = torch.optim.AdamW(model.parameters(), lr=3e-3, weight_decay=0.0); B, T = 32, 64
    snaps = {}; sin_usar = [i for i in range(V) if cnt[i] == 0]
    for paso in range(pasos + 1):
        if paso % 100 == 0: snaps[paso] = model.emb.weight.detach().numpy().copy()
        if paso == pasos: break
        ix = torch.randint(0, len(ids) - T - 1, (B,)); x = torch.stack([ids[i:i + T] for i in ix]); y = torch.stack([ids[i + 1:i + T + 1] for i in ix])
        loss = F.cross_entropy(model(x).reshape(-1, V), y.reshape(-1)); opt.zero_grad(); loss.backward(); opt.step()
    model.eval()
    vocab = tk.get_vocab(); inv = {i: t for t, i in vocab.items()}
    intr_ids = sorted({i for n in intr for i in tk.encode(n).ids if cnt[i] == 0})
    def logits(x):
        with torch.no_grad(): return model(torch.as_tensor(x))
    return {"nombre": f"MiniGPT (tokenizer {V} con intrusos; modelo sin intrusos)", "E": model.emb.weight.detach().numpy().copy(), "E_out": model.emb.weight.detach().numpy().copy(), "tied": True,
            "tokens": [token_a_bytes(inv[i]) if inv[i] != "<|endoftext|>" else b"<|endoftext|>" for i in range(V)], "encode": lambda s: tk.encode(s).ids,
            "decode": lambda ids_: tk.decode(list(ids_)), "logits": logits, "tok": tk, "ctx": model.ctx, "frecuencias": cnt, "intrusos": intr_ids, "snaps": snaps,
            "ids_entrenamiento": ids, "loss_final": float(loss), "modelo": model}

In [ ]:
def bundle_hf(model, tok, nombre: str) -> dict:
    """Envuelve un modelo causal de HF (p. ej. GPT-2) con la MISMA interfaz que banco_mini. tok: AutoTokenizer byte-level."""
    E = model.get_input_embeddings().weight.detach().cpu().numpy(); Eo = model.get_output_embeddings().weight.detach().cpu().numpy()
    V = E.shape[0]; vocab = tok.convert_ids_to_tokens(list(range(V)))
    def logits(x):
        with torch.no_grad(): return model(torch.as_tensor(x)).logits
    return {"nombre": nombre, "E": E, "E_out": Eo, "tied": model.get_input_embeddings().weight is model.get_output_embeddings().weight,
            "tokens": [token_a_bytes(t) if t else b"" for t in vocab], "encode": lambda s: tok.encode(s), "decode": lambda ids_: tok.decode(list(ids_)),
            "logits": logits, "tok": tok, "ctx": getattr(model.config, "n_positions", 1024), "frecuencias": None, "intrusos": None, "snaps": None, "modelo": model}

def cargar_banco(docs, fast_dev: bool = True) -> dict:
    """Cadena: GPT-2 real de HF (si hay red y fast_dev=False o se pide) -> MiniGPT local con intrusos (siempre disponible)."""
    if not fast_dev and hay_red("https://huggingface.co"):
        try:
            from transformers import AutoModelForCausalLM, AutoTokenizer
            m = AutoModelForCausalLM.from_pretrained("gpt2").eval(); t = AutoTokenizer.from_pretrained("gpt2")
            print("Usando GPT-2 real (124M) de Hugging Face."); return bundle_hf(m, t, "GPT-2 (124M)")
        except Exception as e:
            print("   (GPT-2 no disponible:", type(e).__name__, ") -> MiniGPT local")
    print("Usando el banco MiniGPT local (tokenizer con 'intrusos' + modelo que nunca los ve). Entrenando ~1-2 min en CPU...")
    return banco_mini(docs, vocab_size=2000, pasos=500)

In [ ]:
rng = random.Random(SEED); train_docs, test_docs = [], []
for lang in ["es", "en", "code"]:
    pool = [d for d in docs if d["lang"] == lang]; rng.shuffle(pool); cut = int(0.8 * len(pool))
    train_docs += pool[:cut]; test_docs += pool[cut:]
banco = cargar_banco(train_docs, fast_dev=FAST_DEV_RUN); V = len(banco["tokens"])
print(f"{banco['nombre']} | V = {V} | atados: {banco['tied']}")
if banco["frecuencias"] is not None:
    y_verdad = np.array([banco["frecuencias"][i] == 0 for i in range(V)])
    print(f"verdad: {y_verdad.sum()} tokens ({y_verdad.mean():.1%}) con frecuencia 0; {len(banco['intrusos'])} intrusos del hilo de usuarios")

def intenta(f, *a, **k):
    try: return f(*a, **k)
    except NotImplementedError as e: print("⏳ pendiente:", e); return None

def ids_referencia(bundle) -> list:
    """Ids de los tokens de bytes 0xF5-0xFF (jamás en UTF-8 válido): referencia de 'token no usado' (Land & Bartolo 2024)."""
    pos = {b: i for i, b in enumerate(bundle["tokens"]) if len(b) == 1}
    return [pos[bytes([b])] for b in BYTES_INVALIDOS if bytes([b]) in pos]

def textos_alcanzables(bundle: dict, ids) -> list:
    """(Dado) Tokens (bytes) que son UTF-8 válido y que el tokenizer reproduce como UN único token: se pueden 'escribir' como texto."""
    out = []
    for i in ids:
        try: s_ = bundle["tokens"][i].decode("utf-8")
        except UnicodeDecodeError: continue
        if s_.strip() and bundle["encode"](s_) == [i]: out.append(bundle["tokens"][i])
    return out

def max_prob_contexto(bundle, n_ctx: int = 200, largo: int = 48, semilla: int = SEED) -> np.ndarray:
    """(Dado) Verificación por comportamiento: máxima probabilidad de cada token sobre n_ctx contextos reales."""
    ids = bundle.get("ids_entrenamiento")
    if ids is None: ids = torch.tensor(bundle["encode"](" ".join(d["text"] for d in train_docs[:60])))
    r = random.Random(semilla); mp = None
    for _ in range(0, n_ctx, 50):
        starts = [r.randrange(0, len(ids) - largo - 1) for _ in range(50)]
        P = F.softmax(bundle["logits"](torch.stack([ids[s:s + largo] for s in starts])), -1).amax(dim=(0, 1)).numpy()
        mp = P if mp is None else np.maximum(mp, P)
    return mp

## Paso 1 · Indicadores y verificación

**TODO.** Implementa:
- `ind_coseno(bundle)`: distancia coseno $1-\cos(\mathbf e_i,\bar{\mathbf u})$ entre cada fila de `bundle["E_out"]` y la **media** $\bar{\mathbf u}$ de las filas de los tokens de `ids_referencia(bundle)`. *Menor distancia = más sospechoso.*
- `ind_norma(bundle)`: norma L2 de cada fila de `bundle["E"]`.
- `candidatos(bundle, k)`: los `k` ids con menor distancia coseno.
- `confirma(bundle, ids, umbral=0.01)`: de esos ids, los que además tienen `max_prob_contexto` < `umbral` (verificación por comportamiento).

<details><summary>🪜 Pista 1</summary>

Todo es álgebra con NumPy sobre la matriz `E` de forma (V, d): normaliza filas, producto con el vector de referencia normalizado.
</details>
<details><summary>🪜 Pista 2</summary>

`ref = E[ids_referencia(bundle)].mean(0)`; `1 - (E @ ref) / (np.linalg.norm(E, axis=1) * np.linalg.norm(ref))`; `np.argsort(dist)[:k]`; `mp = max_prob_contexto(bundle)`; `[i for i in ids if mp[i] < umbral]`.
</details>

In [ ]:
def ind_coseno(bundle) -> np.ndarray:
    # TODO: distancia coseno de cada fila de E_out a la media de los tokens de referencia
    raise NotImplementedError("TODO paso 1a")

def ind_norma(bundle) -> np.ndarray:
    # TODO: norma L2 de cada fila de la matriz de ENTRADA E
    raise NotImplementedError("TODO paso 1b")

def candidatos(bundle, k: int) -> list:
    # TODO: ids de los k tokens con menor distancia coseno
    raise NotImplementedError("TODO paso 1c")

def confirma(bundle, ids: list, umbral: float = 0.01) -> list:
    # TODO: filtra los ids cuya máxima probabilidad sobre contextos reales sea < umbral
    raise NotImplementedError("TODO paso 1d")

K = 100
cands = intenta(candidatos, banco, K); conf = intenta(confirma, banco, cands) if cands is not None else None
if cands is not None and banco["frecuencias"] is not None:
    print(f"precisión@{K} (frecuencia 0): {y_verdad[cands].mean():.2f} | recall de intrusos en el top-{K}: {len(set(cands) & set(banco['intrusos'])) / len(banco['intrusos']):.2f} | confirmados: {len(conf)}")
elif cands is not None: print([banco["tokens"][i].decode("utf-8", "replace") for i in cands[:20]])

## Paso 2 · Sanitizar Unicode invisible

**TODO.** `inspecciona(texto)` devuelve un dict con `invisibles` (nº de caracteres de categoría `Cf`, `Co`, `Cn`, `Cs` o del bloque Tags U+E0000–U+E007F), `en_tags` y `palabras_mixtas` (palabras que mezclan escrituras LATIN/CYRILLIC/GREEK). `sanitiza(texto)` aplica NFKC y elimina **tags, ancho cero (U+200B, U+2060, U+FEFF), bidi (U+202A–U+202E, U+2066–U+2069)**, pero **conserva ZWJ (U+200D) y VS16 (U+FE0F)** para no romper emojis compuestos.

<details><summary>🪜 Pista 1</summary>

Decide por *code point*: una lista de rangos a eliminar y una lista blanca explícita (200D, FE0F). `unicodedata.name(c).split()[0]` da la escritura de una letra.
</details>
<details><summary>🪜 Pista 2</summary>

`BORRAR = [(0xE0000, 0xE007F), (0x200B, 0x200C), (0x2060, 0x2064), (0xFEFF, 0xFEFF), (0x202A, 0x202E), (0x2066, 0x2069)]`; recorre y conserva el carácter si no cae en ningún rango.
</details>

In [ ]:
def inspecciona(texto: str) -> dict:
    # TODO: {"invisibles": int, "en_tags": int, "palabras_mixtas": list[str]}
    raise NotImplementedError("TODO paso 2a")

def sanitiza(texto: str) -> str:
    # TODO: NFKC + eliminar tags/ancho cero/bidi conservando ZWJ y VS16
    raise NotImplementedError("TODO paso 2b")

def oculta_tags(texto: str) -> str:
    return "".join(chr(0xE0000 + ord(c)) for c in texto if 32 <= ord(c) < 127)
TESTS_SAN = [("pa​ssword", "password"), ("hola" + oculta_tags("mensaje oculto inocuo"), "hola"), ("factura‮txt", "facturatxt"),
             ("familia 👨‍👩‍👧", "familia 👨‍👩‍👧"), ("ﬁnanzas", "finanzas"), ("texto normal", "texto normal")]
res_san = [intenta(sanitiza, a) == b for a, b in TESTS_SAN]; print("tests de sanitiza:", res_san)

## Paso 3 · Fuzzing con propiedades

**TODO.** `fuzz_props(bundle, clases, n, glitch_ids, especiales_ids)` usa el generador `genera(clase, r, glitch_txt)` (te lo damos; `glitch_txt` = `textos_alcanzables(bundle, sorted(glitch_ids))[:50]`) y debe devolver un `DataFrame` con una fila por entrada y columnas booleanas `P1_roundtrip`, `P2_sin_control`, `P3_ratio` (tokens/bytes ≤ 1), `P4_finito` (logits finitos) y `P5_sin_glitch`. (Te damos `textos_alcanzables` para construir las entradas de la clase `glitch`.) Después escribe `informe` (lista de dicts) con **≥ 3 hallazgos** `{"hallazgo", "severidad": "alta|media|baja", "evidencia", "mitigacion"}` apoyados en tus resultados.

<details><summary>🪜 Pista 1</summary>

Para cada entrada: `ids = bundle["encode"](x)`; P1 = `bundle["decode"](ids) == x`; P2 = `not (set(ids) & especiales_ids)`; P3 = `len(ids) / max(len(x.encode()), 1) <= 1`; P4 = `torch.isfinite(bundle["logits"](torch.tensor([ids[-bundle["ctx"]:]]))).all()`; P5 = `not (set(ids) & glitch_ids)`.
</details>

In [ ]:
ESPECIALES_TXT = ["<|endoftext|>", "<|im_start|>", "<|im_end|>", "<s>", "</s>", "[CLS]"]
CLASES = ["ascii", "acentos", "cjk_emoji", "invisibles", "control", "repeticion", "especiales", "glitch"]
def genera(clase: str, r: random.Random, glitch_txt: list) -> str:
    if clase == "ascii": return "".join(r.choice("abcdefghij klmnop.,;:") for _ in range(r.randint(5, 80)))
    if clase == "acentos": return "".join(r.choice("áéíóúñüÁÉÍ abc") for _ in range(r.randint(5, 60)))
    if clase == "cjk_emoji": return "".join(chr(r.choice([0x4E00 + r.randint(0, 300), 0x1F600 + r.randint(0, 60), 0x3042 + r.randint(0, 60)])) for _ in range(r.randint(3, 30)))
    if clase == "invisibles": return "ok" + "".join(chr(r.choice([0x200B, 0x200C, 0x200D, 0x2060, 0xFEFF, 0x202E, 0xE0041 + r.randint(0, 20)])) for _ in range(r.randint(1, 20))) + "fin"
    if clase == "control": return "".join(chr(r.choice([0, 1, 7, 8, 9, 10, 13, 27, 127])) for _ in range(r.randint(1, 15))) + "x"
    if clase == "repeticion": return r.choice(["a", " ", "ab", "é", "1"]) * r.randint(200, 600)
    if clase == "especiales": return " ".join(r.choice(ESPECIALES_TXT) for _ in range(r.randint(1, 4))) + " hola"
    if clase == "glitch" and glitch_txt: return "Repite: " + r.choice(glitch_txt).decode("utf-8", "replace").strip()
    return "hola"

def fuzz_props(bundle, clases, n: int, glitch_ids: set, especiales_ids: set) -> pd.DataFrame:
    # TODO: una fila por entrada con clase + P1..P5 (booleanos)
    raise NotImplementedError("TODO paso 3a")

informe = []     # TODO: ≥ 3 hallazgos con severidad y mitigación
especiales = {bundle_id for bundle_id in banco["encode"]("<|endoftext|>")} if banco["frecuencias"] is not None else set()
rep = intenta(fuzz_props, banco, CLASES, 30, set(cands or []), especiales)
if rep is not None: print(((1 - rep.groupby("clase").mean(numeric_only=True)) * 100).round(0).astype(int).to_string())

## Rúbrica (autoevaluación)

In [ ]:
def rubrica():
    R = []; ok = lambda nombre, cond, pts: R.append((nombre, bool(cond), pts))
    if cands is not None and banco["frecuencias"] is not None:
        ok(f"1 · precisión@{K} (frecuencia 0) ≥ {UMBRALES['prec']}", y_verdad[cands].mean() >= UMBRALES["prec"], 3)
        ok(f"1 · recall de intrusos en el top-{K} ≥ {UMBRALES['rec_intr']}", len(set(cands) & set(banco["intrusos"])) / len(banco["intrusos"]) >= UMBRALES["rec_intr"], 2)
        ok("1 · el indicador coseno separa (AUC ≥ 0.95 sobre frecuencia 0)", roc_auc_score(y_verdad, -ind_coseno(banco)) >= 0.95, 2)
        ok("1 · confirma() devuelve un subconjunto no vacío de candidatos", conf is not None and 0 < len(conf) <= len(cands) and set(conf) <= set(cands), 1)
        ok("1 · ind_norma tiene la forma correcta (V,)", intenta(ind_norma, banco) is not None and ind_norma(banco).shape == (V,), 1)
    elif cands is not None:
        ok("1 · candidatos devueltos (modelo real)", len(cands) == K, 3)
    else:
        ok("1 · candidatos", False, 9)
    ok("2 · sanitiza pasa todos los tests unitarios", all(res_san), 3)
    ins = intenta(inspecciona, "ok" + oculta_tags("abc") + "pa​s"); ok("2 · inspecciona cuenta tags e invisibles", ins is not None and ins["en_tags"] == 3 and ins["invisibles"] >= 4, 1)
    ins2 = intenta(inspecciona, "pаgos"); ok("2 · inspecciona detecta palabras con escrituras mezcladas", ins2 is not None and len(ins2["palabras_mixtas"]) == 1, 1)
    if rep is not None:
        fl = (1 - rep.groupby("clase").mean(numeric_only=True)) * 100
        ok("3 · fuzz_props devuelve las 5 propiedades por clase", {"P1_roundtrip", "P2_sin_control", "P3_ratio", "P4_finito", "P5_sin_glitch"} <= set(rep.columns), 2)
        ok(f"3 · el fuzzer detecta la clase 'glitch' (P5 falla ≥ {UMBRALES['glitch_p5']} %)", fl.loc["glitch", "P5_sin_glitch"] >= UMBRALES["glitch_p5"], 1)
    else:
        ok("3 · fuzz_props", False, 3)
    ok("3 · informe con ≥ 3 hallazgos completos", len(informe) >= 3 and all({"hallazgo", "severidad", "evidencia", "mitigacion"} <= set(h) and h["severidad"] in ("alta", "media", "baja") for h in informe), 2)
    tot = sum(p for _, _, p in R); got = sum(p for _, c, p in R if c)
    for nombre, c, p in R: print(("✅" if c else "⬜"), f"[{p}]", nombre)
    print(f"\nPuntuación: {got}/{tot}")
UMBRALES = {"prec": 0.90, "rec_intr": 0.30, "glitch_p5": 60}
rubrica()

## ⛔ SPOILER — intenta resolverlo primero

Debajo está la **solución de referencia completa**; al ejecutarla se redefinen las funciones y la rúbrica puntúa la referencia.

In [ ]:
def ind_coseno(bundle) -> np.ndarray:
    E = bundle["E_out"]; ref = E[ids_referencia(bundle)].mean(0)
    return 1 - (E @ ref) / (np.linalg.norm(E, axis=1) * np.linalg.norm(ref) + 1e-12)

def ind_norma(bundle) -> np.ndarray:
    return np.linalg.norm(bundle["E"], axis=1)

def candidatos(bundle, k: int) -> list:
    return [int(i) for i in np.argsort(ind_coseno(bundle))[:k]]

def confirma(bundle, ids: list, umbral: float = 0.01) -> list:
    mp = max_prob_contexto(bundle); return [i for i in ids if mp[i] < umbral]

cands = candidatos(banco, K); conf = confirma(banco, cands)
if banco["frecuencias"] is not None:
    print(f"precisión@{K}: {y_verdad[cands].mean():.2f} | recall de intrusos en el top-{K}: {len(set(cands) & set(banco['intrusos'])) / len(banco['intrusos']):.2f} | confirmados: {len(conf)} | AUC coseno: {roc_auc_score(y_verdad, -ind_coseno(banco)):.3f}")
else: print([banco["tokens"][i].decode("utf-8", "replace") for i in cands[:20]])

In [ ]:
BORRAR = [(0xE0000, 0xE007F), (0x200B, 0x200C), (0x2060, 0x2064), (0xFEFF, 0xFEFF), (0x202A, 0x202E), (0x2066, 0x2069)]
def _borrable(c: str) -> bool: return any(a <= ord(c) <= b for a, b in BORRAR)

def inspecciona(texto: str) -> dict:
    inv = [c for c in texto if unicodedata.category(c) in ("Cf", "Co", "Cn", "Cs") or 0xE0000 <= ord(c) <= 0xE007F]
    escr = lambda w: {unicodedata.name(c, "?").split()[0] for c in w if c.isalpha()}
    return {"invisibles": len(inv), "en_tags": sum(0xE0000 <= ord(c) <= 0xE007F for c in texto),
            "palabras_mixtas": [w for w in re.findall(r"\w+", texto) if len(escr(w) & {"LATIN", "CYRILLIC", "GREEK"}) > 1]}

def sanitiza(texto: str) -> str:
    return "".join(c for c in unicodedata.normalize("NFKC", texto) if not _borrable(c))

res_san = [sanitiza(a) == b for a, b in TESTS_SAN]; print("tests de sanitiza:", res_san)

In [ ]:
def fuzz_props(bundle, clases, n: int, glitch_ids: set, especiales_ids: set) -> pd.DataFrame:
    r = random.Random(SEED); glitch_txt = textos_alcanzables(bundle, sorted(glitch_ids))[:50]; filas = []
    for clase in clases:
        for _ in range(n):
            x = genera(clase, r, glitch_txt); ids = bundle["encode"](x)
            try: p1 = bundle["decode"](ids) == x
            except Exception: p1 = False
            try: p4 = bool(torch.isfinite(bundle["logits"](torch.tensor([ids[-bundle["ctx"]:]]))).all()) if ids else True
            except Exception: p4 = False
            filas.append({"clase": clase, "P1_roundtrip": p1, "P2_sin_control": not (set(ids) & especiales_ids), "P3_ratio": len(ids) / max(len(x.encode()), 1) <= 1 + 1e-9,
                          "P4_finito": p4, "P5_sin_glitch": not (set(ids) & glitch_ids)})
    return pd.DataFrame(filas)

rep = fuzz_props(banco, CLASES, 30, set(cands), especiales)
fl = (1 - rep.groupby("clase").mean(numeric_only=True)) * 100; print(fl.round(0).astype(int).to_string())
informe = [
    {"hallazgo": "Tokens sub-entrenados (intrusos del hilo de usuarios) presentes en el vocabulario", "severidad": "alta",
     "evidencia": f"precisión@{K} de {y_verdad[cands].mean():.2f} con el indicador coseno; {len(conf)} candidatos confirmados por comportamiento" if banco["frecuencias"] is not None else "candidatos por coseno",
     "mitigacion": "Alinear el corpus del tokenizer con el del modelo; eliminar o re-inicializar esas filas; filtrar entradas que los contengan; auditoría en CI"},
    {"hallazgo": "Los textos con especiales (<|endoftext|>) generan el token de control real", "severidad": "alta",
     "evidencia": f"P2 falla en {fl.loc['especiales', 'P2_sin_control']:.0f} % de las entradas de la clase 'especiales'",
     "mitigacion": "Tokenizar contenido de usuario con especiales desactivados (split_special_tokens / disallowed_special) y construir la plantilla fuera del texto del usuario"},
    {"hallazgo": "Texto invisible y caracteres de control llegan intactos al modelo", "severidad": "media",
     "evidencia": f"P5 falla en {fl.loc['invisibles', 'P5_sin_glitch']:.0f} % de las entradas 'invisibles': los bytes de esos caracteres son tokens sub-entrenados y cada carácter oculto cuesta varios tokens",
     "mitigacion": "sanitiza() antes de indexar y de promptear; ratio tokens/caracteres visibles como alarma"},
]
for h in informe: print(f"[{h['severidad']}] {h['hallazgo']}\n    evidencia: {h['evidencia']}\n    mitigación: {h['mitigacion']}")
rubrica()

## Retos extra (nivel experto)

1. **Umbral automático.** En lugar de `k` fijo, elige el corte con el *codo* de la distribución de distancias coseno (p. ej. máxima segunda derivada) y compáralo con el `k` óptimo según la verdad.
2. **Dos indicadores.** Combina coseno y norma (z-scores) y mide si mejora la precisión; ¿qué pasa si el modelo tiene embeddings **no atados**? (entrena una variante con `E_out` independiente).
3. **GPT-2 real.** Con `FAST_DEV_RUN=False` y red, lista los 50 mejores candidatos, comprueba si ` SolidGoldMagikarp`, ` TheNitromeFan` o ` RandomRedditor` son un único token y su rango, y verifica con prompts de repetición.
4. **Parciales y alcanzables.** Añade la auditoría de tokens parciales e inalcanzables (lección) al informe del tokenizer propio de Políglota (proyectos 03–04).
5. **Healing en Políglota.** Mide la masa compatible con/sin *token healing* en consultas del buscador cortadas a mitad de palabra (autocompletado).

## Reflexión (producción y Políglota)

- ¿Qué parte de este informe automatizarías en CI y con qué umbrales para cada nueva versión del tokenizer?
- Un documento de un cliente trae 20 KB de texto invisible. ¿Dónde lo detectas (ingesta, indexado, consulta, prompt) y qué registras para auditoría?
- ¿Qué patologías de esta lección **desaparecen** con un modelo a nivel de byte (módulo 06) y cuáles aparecen en su lugar?